# #7DaysOfCode — Agentes de IA & Engenharia de Prompts
## Resolução Técnica — Dia 4: O Ciclo ReAct (`Thought -> Action -> Observation`)

Este notebook apresenta a implementação técnica do **Dia 4**, estruturando o motor de raciocínio e ação passo a passo (**Ciclo ReAct**) da assistente **Clara** e adicionando a verificação de disponibilidade no estoque do Centro de Distribuição (`ESTOQUE_CD_NOVAMART`).

### Escopo Técnico desta Implementação
- Criação da 3ª ferramenta operacional (`@tool verificar_estoque_produto_cd`) integrada à tabela `ESTOQUE_CD_NOVAMART`;
- Engenharia do `SYSTEM_PROMPT_REACT_NOVAMART` para explicitar traços de raciocínio (`PENSAMENTO`) antes de cada invocação de ferramenta (`AÇÃO`) e após cada retorno (`OBSERVAÇÃO`);
- Implementação do executor autônomo multi-etapas `executar_agente_react_dia4( Assegurando limite de segurança max_iter=6)`.

---

## Etapa 1: Preparação do Ambiente e Conexão Segura

Instalamos as bibliotecas da stack (`langchain`, `langchain-core`, `langchain-google-genai`, `pydantic` e `python-dotenv`) e configuramos a `GOOGLE_API_KEY`.

In [ ]:
# Nota: No Google Colab, pode aparecer um aviso sobre versão do 'google-auth', mas a instalação funciona normalmente!
!pip install -q -U langchain langchain-core langchain-google-genai pydantic python-dotenv

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("Chave GOOGLE_API_KEY carregada com sucesso dos Secrets do Google Colab!")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Digite sua GOOGLE_API_KEY (Google AI Studio): ")
        print("Chave configurada com sucesso para esta sessão!")
else:
    print("GOOGLE_API_KEY já está configurada no ambiente!")

---
## Etapa 2: Base Herdada (Dias 1 a 3) + Tabela de Estoque do CD (`ESTOQUE_CD_NOVAMART`)

Na célula abaixo, carregamos:
- O modelo **`gemini-2.5-flash`** (`temperature=0.0`);
- O **`BANCO_PEDIDOS_NOVAMART`** lido do arquivo externo [`../data/pedidos_novamart.json`](../data/pedidos_novamart.json) e as ferramentas construídas no **Dia 3** (`consultar_pedido_novamart` e `abrir_protocolo_logistica_erp`);
- A tabela **`ESTOQUE_CD_NOVAMART`**, que representa a disponibilidade de produtos e tamanhos no Centro de Distribuição da NovaMart.

In [ ]:
import json
from pathlib import Path
from typing import Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

# Inicializando o modelo Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.0
)

# =====================================================================
# HERANÇA DO DIA 3: Banco de Pedidos Oficial (pedidos_novamart.json)
# =====================================================================
def localizar_arquivo_dados(nome_arquivo: str) -> Path:
    """Localiza o arquivo na pasta '../data/', 'data/' ou no diretório atual do Colab."""
    for candidato in [Path(f"../data/{nome_arquivo}"), Path(f"data/{nome_arquivo}"), Path(nome_arquivo)]:
        if candidato.exists():
            return candidato
    raise FileNotFoundError(
        f"Arquivo '{nome_arquivo}' não encontrado! Certifique-se de que ele está em '../data/{nome_arquivo}' "
        f"ou faça o upload de '{nome_arquivo}' na aba de Arquivos do Google Colab."
    )

caminho_pedidos = localizar_arquivo_dados("pedidos_novamart.json")
BANCO_PEDIDOS_NOVAMART = json.loads(caminho_pedidos.read_text(encoding="utf-8"))

# =====================================================================
# NOVIDADE DO DIA 4: Tabela de Estoque do Centro de Distribuição (CD)
# =====================================================================
ESTOQUE_CD_NOVAMART = {
    "TENIS_ULTRABOOST_ROSA_37": {
        "sku": "TENIS_ULTRABOOST_ROSA_37",
        "produto": "Tênis de Corrida UltraBoost Rosa (Tam 37)",
        "unidades_disponiveis": 8,
        "status_estoque": "DISPONIVEL"
    },
    "BOTA_COURO_PRETA_39": {
        "sku": "BOTA_COURO_PRETA_39",
        "produto": "Bota de Couro Cano Curto Preta (Tam 39)",
        "unidades_disponiveis": 0,
        "status_estoque": "ESGOTADO"
    },
    "CAFETEIRA_BARISTA_PRO": {
        "sku": "CAFETEIRA_BARISTA_PRO",
        "produto": "Cafeteira Expresso Barista Pro 15 Bar",
        "unidades_disponiveis": 5,
        "status_estoque": "DISPONIVEL"
    }
}

PROTOCOLOS_REGISTRADOS_ERP = []

print(f"Bases carregadas de '{caminho_pedidos}': {len(BANCO_PEDIDOS_NOVAMART)} pedidos no ERP e {len(ESTOQUE_CD_NOVAMART)} SKUs no Estoque do CD!")

---
## Etapa 3: Criando a 3ª Ferramenta Operacional (`verificar_estoque_produto_cd`)

Além de `consultar_pedido_novamart` e `abrir_protocolo_logistica_erp`, definimos a ferramenta **`verificar_estoque_produto_cd`**.

### Papel da verificação de estoque no fluxo de decisão
Quando um cliente solicita **troca de tamanho ou cor** (ex: pedido `#NVM-3091` para o tamanho 39), o agente deve consultar a disponibilidade do item no CD antes de emitir o protocolo:
- Se o SKU estiver **`DISPONIVEL`**, o agente pode registrar o protocolo `POSTAGEM_GRATUITA_TROCA_DEVOLUCAO`;
- Se o SKU estiver **`ESGOTADO`** (`0` unidades), o agente não abre protocolo de troca direta sem o consentimento do cliente, informando a indisponibilidade e oferecendo **Vale-Troca**, troca por outro modelo ou reembolso (quando aplicável).

In [ ]:
import unicodedata

def _normalizar_texto(texto: str) -> str:
    """Remove acentos e converte para maiúsculas para facilitar a busca no estoque."""
    sem_acento = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return sem_acento.upper().strip()

@tool
def consultar_pedido_novamart(codigo_pedido: str) -> dict:
    """
    Consulta o banco de dados oficial da NovaMart para obter os dados reais de um pedido.
    Use esta ferramenta SEMPRE que o cliente mencionar um código de pedido (ex: '#NVM-4821', 'NVM-3091' ou '4821').
    Retorna nome do cliente, nível de fidelidade (Padrão ou VIP Prime), produto, categoria,
    status da entrega ('ENTREGUE' ou 'EM_TRANSITO') e o número exato de dias corridos desde a entrega.
    """
    codigo_limpo = codigo_pedido.strip().upper()
    if not codigo_limpo.startswith("#"):
        codigo_limpo = f"#{codigo_limpo}" if codigo_limpo.startswith("NVM-") else f"#NVM-{codigo_limpo}"

    pedido = BANCO_PEDIDOS_NOVAMART.get(codigo_limpo)
    if not pedido:
        return {
            "encontrado": False,
            "codigo_pesquisado": codigo_limpo,
            "mensagem_sistema": f"Pedido {codigo_limpo} não localizado na base de dados da NovaMart."
        }
    return {"encontrado": True, "dados_pedido": pedido}


@tool
def verificar_estoque_produto_cd(termo_busca_produto: str) -> dict:
    """
    Consulta o estoque do Centro de Distribuição (CD) da NovaMart para verificar se um produto,
    modelo ou tamanho específico está DISPONIVEL ou ESGOTADO para substituição/troca.
    Use SEMPRE antes de abrir um protocolo de troca de tamanho/cor para confirmar se o item desejado
    pelo cliente (ex: 'Bota Preta 39', 'Tênis Rosa 37', 'BOTA_COURO_PRETA_39') tem unidades disponíveis.
    """
    busca_norm = _normalizar_texto(termo_busca_produto)
    # Removemos palavras genéricas de ligação para focar nas palavras-chave do produto e tamanho
    stopwords = {"DE", "DO", "DA", "NO", "NA", "PARA", "POR", "TAMANHO", "TAM", "NUMERO", "COR", "MODELO"}
    palavras_chave = [p for p in busca_norm.replace("_", " ").split() if p not in stopwords]

    # 1. Busca direta por SKU exato
    if busca_norm in ESTOQUE_CD_NOVAMART:
        return {"encontrado": True, "item_estoque": ESTOQUE_CD_NOVAMART[busca_norm]}

    # 2. Busca por correspondência de palavras-chave no SKU ou nome do produto
    melhor_item = None
    maior_pontuacao = 0
    for sku, dados in ESTOQUE_CD_NOVAMART.items():
        alvo = _normalizar_texto(f"{sku} {dados['produto']}")
        pontos = sum(1 for p in palavras_chave if p in alvo)
        if pontos > maior_pontuacao:
            maior_pontuacao = pontos
            melhor_item = dados

    if melhor_item and maior_pontuacao > 0:
        return {
            "encontrado": True,
            "item_estoque": melhor_item,
            "orientacao_logistica": (
                "Produto disponível para reserva e troca imediata."
                if melhor_item["status_estoque"] == "DISPONIVEL"
                else "ATENÇÃO: Tamanho/modelo ESGOTADO no CD (0 unidades). Não abra protocolo de troca direta para este tamanho; ofereça Vale-Troca (liberado em até 24h após postagem) ou escolha de outro modelo/numeração."
            )
        }

    return {
        "encontrado": False,
        "termo_pesquisado": termo_busca_produto,
        "itens_cadastrados_no_cd": list(ESTOQUE_CD_NOVAMART.values())
    }


@tool
def abrir_protocolo_logistica_erp(
    codigo_pedido: str,
    tipo_operacao: Literal["POSTAGEM_GRATUITA_TROCA_DEVOLUCAO", "ORDEM_ANALISE_TECNICA_CD"],
    motivo_resumido: str
) -> dict:
    """
    Abre um protocolo oficial no sistema ERP de logística reversa da NovaMart e gera código de postagem dos Correios.
    IMPORTANTE:
    1. Só chame esta ferramenta DEPOIS de consultar o pedido em 'consultar_pedido_novamart' e confirmar que é ELEGÍVEL.
    2. Se for troca por novo tamanho/modelo, verifique ANTES em 'verificar_estoque_produto_cd' se o item desejado está DISPONIVEL.
       Se estiver ESGOTADO e o cliente ainda não tiver autorizado Vale-Troca, NÃO abra o protocolo ainda: informe o esgotamento e pergunte qual opção ele prefere.
    3. Se for defeito funcional/técnico entre 8 e 90 dias corridos da entrega, use tipo_operacao='ORDEM_ANALISE_TECNICA_CD'.
    """
    codigo_limpo = codigo_pedido.strip().upper()
    if not codigo_limpo.startswith("#"):
        codigo_limpo = f"#{codigo_limpo}" if codigo_limpo.startswith("NVM-") else f"#NVM-{codigo_limpo}"

    pedido = BANCO_PEDIDOS_NOVAMART.get(codigo_limpo)
    if not pedido or pedido["status_entrega"] != "ENTREGUE":
        return {
            "sucesso": False,
            "erro": "Não é possível abrir protocolo de logística reversa para pedido inexistente ou ainda não entregue."
        }

    numero_sequencial = len(PROTOCOLOS_REGISTRADOS_ERP) + 401
    num_digits = codigo_limpo.replace("#NVM-", "")
    protocolo_id = f"PROT-2026-{numero_sequencial}"
    codigo_correios = f"BR-POST-{num_digits}-{numero_sequencial}"

    registro = {
        "protocolo": protocolo_id,
        "codigo_pedido": codigo_limpo,
        "cliente": pedido["cliente"],
        "nivel_fidelidade": pedido["nivel_fidelidade"],
        "tipo_operacao": tipo_operacao,
        "codigo_postagem_correios": codigo_correios,
        "motivo": motivo_resumido,
        "beneficio_vip_aplicado": pedido["nivel_fidelidade"] == "VIP Prime"
    }
    PROTOCOLOS_REGISTRADOS_ERP.append(registro)
    return {"sucesso": True, "detalhes_protocolo": registro}

# Consolidando o kit de 3 ferramentas do Dia 4
ferramentas_dia4 = [
    consultar_pedido_novamart,
    verificar_estoque_produto_cd,
    abrir_protocolo_logistica_erp
]
mapa_ferramentas_dia4 = {f.name: f for f in ferramentas_dia4}

# Teste rápido da nova ferramenta de estoque:
print("Teste rápido da Tool 'verificar_estoque_produto_cd':")
print(" -> Busca 'Bota Preta Tam 39':", verificar_estoque_produto_cd.invoke({"termo_busca_produto": "Bota Preta Tam 39"})["item_estoque"])
print(" -> Busca 'Tênis Rosa Tam 37':", verificar_estoque_produto_cd.invoke({"termo_busca_produto": "Tênis Rosa Tam 37"})["item_estoque"])

---
## Etapa 4: Implementando o Prompt e o Motor do Ciclo ReAct (`Pensamento -> Ação -> Observação`)

O paradigma **ReAct (*Reasoning + Acting*)** (*Yao et al.*) combina geração explícita de raciocínio verbal (*Thought*) com chamadas de ferramentas (*Action*) e análise dos resultados retornados (*Observation*):

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│                        O CICLO REACT NA NOVAMART                            │
│                                                                             │
│   [Mensagem do Cliente]                                                     │
│             │                                                               │
│             ▼                                                               │
│   ┌───────────────────┐                                                     │
│   │ 1. PENSAMENTO     │ -> "Preciso consultar o pedido #NVM-3091 para ver   │
│   │    (Thought)      │     quantos dias tem de entrega e o nível VIP..."   │
│   └─────────┬─────────┘                                                     │
│             │                                                               │
│             ▼                                                               │
│   ┌───────────────────┐                                                     │
│   │ 2. AÇÃO           │ -> Chama @tool: consultar_pedido_novamart('#NVM-3091')│
│   │    (Action)       │                                                     │
│   └─────────┬─────────┘                                                     │
│             │                                                               │
│             ▼                                                               │
│   ┌───────────────────┐                                                     │
│   │ 3. OBSERVAÇÃO     │ -> Retorno JSON: 12 dias, Padrão, Bota Preta Tam 38 │
│   │    (Observation)  │    (Retorna ao Pensamento para checar o estoque!)   │
│   └─────────┬─────────┘                                                     │
│             │                                                               │
│             ▼  (Repete até concluir todas as verificações)                  │
│   [Resposta Final Consolidada para o Cliente]                               │
└─────────────────────────────────────────────────────────────────────────────┘
```

Definimos o `SYSTEM_PROMPT_REACT_NOVAMART` e o motor **`executar_agente_react_dia4()`** com `max_iter=6` e log detalhado em console.

In [ ]:
import json
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

SYSTEM_PROMPT_REACT_NOVAMART = """
Você é a Clara, Agente Autônoma Sênior de Atendimento e Logística Reversa da NovaMart, operando no paradigma ReAct (Reasoning + Acting).

<regras_de_negocio>
1. DIREITO DE ARREPENDIMENTO (Devolução com Reembolso Integral): Até 7 dias corridos após o recebimento (sem uso, na caixa original e com lacre/etiqueta intactos).
2. TROCA DE TAMANHO OU COR (Moda, Calçados e Acessórios):
   - Cliente 'Padrão': Prazo de até 30 dias corridos após o recebimento.
   - Cliente 'VIP Prime': Prazo estendido de até 60 dias corridos após o recebimento + liberação expressa no ato da postagem.
   - Condição obrigatória: Produto sem uso e com etiqueta afixada. A 1ª troca é gratuita.
   - REGRA DE ESTOQUE DO CD: Antes de abrir protocolo para troca de tamanho/modelo, você DEVE consultar 'verificar_estoque_produto_cd'.
     * Se o tamanho desejado estiver 'DISPONIVEL', abra o protocolo com tipo_operacao='POSTAGEM_GRATUITA_TROCA_DEVOLUCAO'.
     * Se o tamanho desejado estiver 'ESGOTADO' (0 unidades), NÃO abra o protocolo de troca direta para esse tamanho! Explique ao cliente que o pedido é elegível dentro do prazo, mas que o tamanho desejado está esgotado no Centro de Distribuição, e ofereça as opções: (a) Vale-Troca no valor integral (liberado em até 24h após a postagem, ou imediato se VIP Prime) para comprar qualquer outro item no site, ou (b) troca por outro modelo/numeração disponível, perguntando como ele prefere prosseguir.
3. DEFEITO TÉCNICO OU DE FABRICAÇÃO:
   - Até 7 dias corridos do recebimento: Troca imediata.
   - De 8 a 90 dias corridos do recebimento: O produto deve ser encaminhado para análise técnica no Centro de Distribuição (prazo de laudo de até 15 dias úteis). Nesse caso, abra o protocolo usando obrigatoriamente tipo_operacao='ORDEM_ANALISE_TECNICA_CD'.
4. AVARIA NO TRANSPORTE: Deve ser reportada em até 48 horas (2 dias corridos) após a entrega.
5. ITENS COM RESTRIÇÃO: Roupas íntimas, moda praia sem protetor, perfumes/cosméticos com lacre violado, personalizados e gift cards não têm troca por insatisfação.
</regras_de_negocio>

<protocolo_react_obrigatorio>
Você opera em um ciclo estruturado de PENSAMENTO -> AÇÃO -> OBSERVAÇÃO:
- ANTES de acionar qualquer ferramenta (@tool), escreva SEMPRE no conteúdo de texto da sua mensagem um parágrafo curto iniciado exatamente por 'PENSAMENTO:' explicando:
  (1) O que você já descobriu até agora;
  (2) O que ainda falta verificar nos sistemas (Pedidos, Estoque do CD ou Protocolo ERP) e qual regra de negócio está avaliando.
- Você pode planejar e executar etapas sequenciais (por exemplo: primeiro consultar os pedidos, no turno seguinte consultar o estoque dos pedidos elegíveis, e depois abrir o protocolo apenas para o pedido elegível e com estoque).
- Quando já tiver todas as observações necessárias e nenhuma ferramenta pendente, NÃO escreva 'PENSAMENTO:': escreva diretamente a Resposta Final Consolidada para o cliente, de forma empática, clara, citando os dados oficiais verificados e os números de protocolo/postagem gerados.
</protocolo_react_obrigatorio>
"""

# Acoplando as 3 ferramentas ao modelo
llm_react = llm.bind_tools(ferramentas_dia4)

def _extrair_texto_pensamento(conteudo) -> str:
    """Extrai o texto limpo de AIMessage.content (seja string ou lista de blocos do Gemini)."""
    if isinstance(conteudo, str):
        return conteudo.strip()
    if isinstance(conteudo, list):
        partes = []
        for bloco in conteudo:
            if isinstance(bloco, dict) and bloco.get("type") == "text":
                partes.append(bloco.get("text", ""))
            elif isinstance(bloco, str):
                partes.append(bloco)
        return "\n".join(partes).strip()
    return str(conteudo).strip()


def executar_agente_react_dia4(
    mensagem_cliente: str,
    max_iter: int = 6,
    verbose: bool = True
) -> dict:
    """
    Motor do Ciclo ReAct (Dia 4): Executa múltiplas iterações autônomas de
    Pensamento (Thought) -> Ação (Action) -> Observação (Observation)
    até consolidar a resposta final para o cliente.
    """
    mensagens = [
        SystemMessage(content=SYSTEM_PROMPT_REACT_NOVAMART),
        HumanMessage(content=mensagem_cliente)
    ]
    trilha_auditoria_react = []

    if verbose:
        print(f"🗣️ [CLIENTE]: {mensagem_cliente}")
        print("-" * 95)

    for iteracao in range(1, max_iter + 1):
        resposta_ia = llm_react.invoke(mensagens)
        mensagens.append(resposta_ia)

        texto_ia = _extrair_texto_pensamento(resposta_ia.content)

        # Se não há chamadas de ferramentas pendentes, chegamos à Resposta Final Consolidada!
        if not resposta_ia.tool_calls:
            if verbose:
                print(f"\n✅ [RESPOSTA FINAL CONSOLIDADA (Iteração {iteracao})]:")
                print(texto_ia)
            return {
                "iteracoes_utilizadas": iteracao,
                "trilha_react": trilha_auditoria_react,
                "resposta_final": texto_ia
            }

        # Se há chamadas de ferramentas, estamos em um passo de Raciocínio + Ação
        pensamento_passo = texto_ia if texto_ia else (
            f"Planejando acionar {[c['name'] for c in resposta_ia.tool_calls]} para coletar dados oficiais antes de decidir."
        )
        if verbose:
            print(f"\n🧠 [ITERAÇÃO {iteracao} — PENSAMENTO (Thought)]:")
            print(f"   {pensamento_passo}")

        # Executando cada ferramenta solicitada pela Clara neste passo
        for chamada in resposta_ia.tool_calls:
            nome_tool = chamada["name"]
            args_tool = chamada["args"]

            if verbose:
                print(f"🛠️ [ITERAÇÃO {iteracao} — AÇÃO (Action)]:")
                print(f"   -> Tool: {nome_tool}({json.dumps(args_tool, ensure_ascii=False)})")

            resultado_tool = mapa_ferramentas_dia4[nome_tool].invoke(args_tool)

            if verbose:
                print(f"👁️ [ITERAÇÃO {iteracao} — OBSERVAÇÃO (Observation)]:")
                print(f"   <- Retorno: {json.dumps(resultado_tool, ensure_ascii=False)}")

            trilha_auditoria_react.append({
                "iteracao": iteracao,
                "pensamento": pensamento_passo,
                "acao_ferramenta": nome_tool,
                "argumentos": args_tool,
                "observacao_retorno": resultado_tool
            })

            mensagens.append(
                ToolMessage(
                    content=json.dumps(resultado_tool, ensure_ascii=False),
                    tool_call_id=chamada["id"],
                    name=nome_tool
                )
            )

    # Caso atinja o limite de iterações, solicita o fechamento final
    resposta_fechamento = llm.invoke(mensagens)
    texto_final = _extrair_texto_pensamento(resposta_fechamento.content)
    return {
        "iteracoes_utilizadas": max_iter,
        "trilha_react": trilha_auditoria_react,
        "resposta_final": texto_final
    }

print("Motor 'executar_agente_react_dia4' configurado com sucesso!")

---
## Etapa 5: Bateria de Validação Multi-Etapas do Ciclo ReAct

Executamos o Agente ReAct nos **3 cenários de validação**:

1. **Caso 1 (`#NVM-3091` — Elegível no Prazo, mas Tamanho Esgotado no CD):**
   - Solicitação de troca da bota preta (Tam 38) pelo **tamanho 39**.
   - Fluxo esperado: consultar `#NVM-3091` (12 dias corridos = dentro do prazo), verificar o estoque da Bota Preta Tam 39 (`ESGOTADO`), não abrir protocolo de troca direta e oferecer alternativas (Vale-Troca ou outro modelo).

2. **Caso 2 (`#NVM-4821` e `#NVM-5502` — Chamado Duplo na Mesma Mensagem):**
   - Solicitação simultânea de troca do tênis rosa do `#NVM-4821` pelo tamanho 37 e da jaqueta corta-vento do `#NVM-5502`.
   - Fluxo esperado: consultar ambos os pedidos, aprovar `#NVM-4821` (*VIP Prime*, 45 dias $\le$ 60, Tam 37 `DISPONIVEL` no CD) abrindo protocolo, e recusar `#NVM-5502` (*Padrão*, 42 dias $>$ 30).

3. **Caso 3 (`#NVM-8840` — Defeito Técnico aos 25 dias $\rightarrow$ Ordem de Análise Técnica no CD):**
   - Relato de falha na bomba de pressão da Cafeteira Expresso (`#NVM-8840`).
   - Fluxo esperado: consultar `#NVM-8840` (25 dias corridos, faixa de 8 a 90 dias para defeito técnico) e registrar protocolo com `tipo_operacao="ORDEM_ANALISE_TECNICA_CD"`.

In [ ]:
# Limpando a lista de protocolos para auditar exatamente o que será aberto na bateria do Dia 4
PROTOCOLOS_REGISTRADOS_ERP.clear()

casos_react_dia4 = [
    # Caso 1: Pedido elegível (12 dias), mas o tamanho desejado (Tam 39) está ESGOTADO no estoque!
    "Oi Clara! Quero trocar minha bota do pedido #NVM-3091 pelo tamanho 39, pois a 38 ficou apertada. Ela está na caixa sem uso e com etiqueta. Se tiver a 39 em estoque, já gera o código de postagem! Se estiver esgotada, me avise quais são minhas opções.",
    
    # Caso 2: Chamado duplo (2 pedidos diferentes na mesma mensagem exigindo raciocínio separado)
    "Olá! Preciso resolver dois pedidos de uma vez: no #NVM-4821 quero trocar meu tênis rosa pelo tamanho 37 (está sem uso na caixa, veja se tem estoque e gere a postagem). E no pedido #NVM-5502 quero trocar a jaqueta corta-vento por outro tamanho.",
    
    # Caso 3: Defeito técnico após 25 dias (Cafeteira #NVM-8840) -> Exige Ordem de Análise Técnica no CD
    "Oi Clara, minha cafeteira do pedido #NVM-8840 parou de funcionar a bomba de pressão hoje. Já pode abrir a ordem de envio para análise técnica no CD para mim?"
]

resultados_auditoria_dia4 = []

for idx, chamado in enumerate(casos_react_dia4, 1):
    print("=" * 95)
    print(f"🚀 EXECUTANDO CASO DE TESTE REACT #{idx}")
    print("=" * 95)
    resultado = executar_agente_react_dia4(chamado, max_iter=6, verbose=True)
    resultados_auditoria_dia4.append(resultado)
    print("\n")

### Auditoria Final do ERP após os 3 Casos do Ciclo ReAct

Vamos inspecionar a tabela `PROTOCOLOS_REGISTRADOS_ERP` para comprovar que:
1. O pedido **`#NVM-3091`** (Bota Tam 39 esgotada) **NÃO** gerou protocolo indevido de troca direta;
2. O pedido **`#NVM-5502`** (Jaqueta com 42 dias em conta Padrão) **NÃO** gerou protocolo;
3. Apenas o **`#NVM-4821`** (`POSTAGEM_GRATUITA_TROCA_DEVOLUCAO`, VIP Prime, Tênis Tam 37 em estoque) e o **`#NVM-8840`** (`ORDEM_ANALISE_TECNICA_CD`, Cafeteira com defeito aos 25 dias) tiveram protocolos abertos!

In [ ]:
print(f"Total de protocolos abertos no ERP durante a bateria do Dia 4: {len(PROTOCOLOS_REGISTRADOS_ERP)}\n")
print(json.dumps(PROTOCOLOS_REGISTRADOS_ERP, indent=2, ensure_ascii=False))

---
## Resumo da Implementação — Dia 4 Concluído

### Componentes construídos neste notebook:
- Implementação da 3ª ferramenta operacional (**`verificar_estoque_produto_cd`**) conectada ao `ESTOQUE_CD_NOVAMART`;
- Estruturação do **Ciclo ReAct (`Thought -> Action -> Observation`)** com log auditável;
- Validação de fluxos multi-etapas cruzando prazo de entrega, nível de fidelidade, disponibilidade de estoque e abertura seletiva de protocolos no ERP.

Compartilhe seu progresso nas redes sociais com a hashtag **#7DaysOfCode**!